# 12.10 · RLHF 与 DPO / RLHF & DPO (Alignment)

> **课程定位 / Where this fits**
> 第 10 课，**Part 12**。把"会说话的模型"变成"听话、有用、无害的助手"的关键——**对齐(alignment)**。
> Lesson 10, **Part 12**. The key to turning a "model that talks" into a "helpful, honest, harmless assistant" — **alignment**.
>
> 预训练 + 微调让模型**会续写文本**，但它**不知道该输出什么样的回答**——可能啰嗦、跑题、编造、甚至有害。**RLHF(基于人类反馈的强化学习)** 是 ChatGPT 成功的核心：用**人类对回答的偏好**来训练模型，让它输出人类更喜欢的内容。它分三步——监督微调(SFT)、训练**奖励模型**、用**强化学习(PPO)** 优化。后来更简单稳定的 **DPO** 跳过 RL、直接用偏好数据优化。本课讲清对齐的动机与流程，并**从零实现奖励模型(偏好学习)** 这一核心组件。
> Pretrain + fine-tune make a model **continue text**, but it **doesn't know what kind of answer to give** — it may ramble, go off-topic, fabricate, or be harmful. **RLHF (Reinforcement Learning from Human Feedback)** is core to ChatGPT's success: train the model on **human preferences over answers** so it outputs what people prefer. Three steps — supervised fine-tuning (SFT), a **reward model**, and **RL (PPO)** optimization. The simpler, stabler **DPO** later skipped RL, optimizing directly on preference data. We cover the motivation and pipeline, and **implement the reward model (preference learning) from scratch** — its core component.
>
> 💼 **实战/面试视角**："RLHF 三步流程 / 奖励模型怎么训 / PPO 的KL惩罚 / DPO vs RLHF" 是 LLM 对齐必考。
> 💼 **Practical/interview angle:** "RLHF's three steps / how the reward model is trained / PPO's KL penalty / DPO vs RLHF" — alignment essentials.

> 📐 **符号约定 / Notation**
> - 偏好对 —— (prompt, 较好回答 chosen, 较差回答 rejected) / a preference pair
> - 奖励模型 $r_\theta$ —— 给(prompt, 回答)打分 / scores a (prompt, response)
> - 策略 policy —— 被优化的语言模型 / the LM being optimized

> 💡 **面试相关 / Interview-relevant**
> - "RLHF 的三个阶段"（出镜率 ★★★★★）
> - "奖励模型用什么损失(Bradley-Terry)"（★★★★）
> - "PPO 里为什么要 KL 惩罚"（★★★★，防跑偏/reward hacking）
> - "DPO 相比 RLHF 的优势"（★★★★★，无需奖励模型和RL）

---

## 学习目标 / Learning Objectives
1. 理解对齐问题与 RLHF 三步流程。
   Understand alignment and RLHF's three steps.
2. **从零实现奖励模型**(从偏好对学习打分)。
   Implement a reward model from scratch (learn to score from preferences).
3. 理解 PPO 优化与 KL 惩罚的作用。
   Understand PPO optimization and the KL penalty.
4. 理解 DPO 如何简化对齐。
   Understand how DPO simplifies alignment.

## 目录 / TOC
1. [对齐问题与 RLHF 三步 ⭐](#1)
2. [奖励模型：从人类偏好学习（从零）⭐](#2)
3. [PPO：用奖励优化策略 ⭐](#3)
4. [DPO：更简单的对齐 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 对齐问题与 RLHF 三步 ⭐ / Alignment & RLHF's Three Steps

**对齐问题**：一个只做过"预测下一个 token"的基础模型，给它 "如何学好数学？" 可能续写成 "如何学好物理？如何学好化学？"——因为训练语料里这种列表很常见。它**没学会"回答问题"这件事本身**，更别说"有用、诚实、无害(helpful, honest, harmless)"。
**The alignment problem:** a base model trained only to "predict the next token," given "How to learn math well?" might continue "How to learn physics well? How to learn chemistry well?" — because such lists are common in training text. It **hasn't learned to "answer"** at all, let alone be "helpful, honest, harmless."

**RLHF 三步流程**(面试必背)：
**RLHF's three steps** (must-know):
1. **监督微调(SFT)**：用人写的"指令→理想回答"示范数据微调基础模型，让它先**学会"回答问题"的格式**(这就是指令微调)。
   **SFT:** fine-tune on human "instruction→ideal answer" demos so it **learns the "answer a question" format** (instruction tuning).
2. **训练奖励模型(RM)**：让人对模型的**多个回答排序**(哪个更好)，用这些**偏好数据**训练一个**奖励模型**，学会给回答**打分**(预测人类偏好)。
   **Train a reward model:** have humans **rank multiple model answers**; train a **reward model** on these **preferences** to **score** answers (predict human preference).
3. **强化学习优化(PPO)**：把语言模型当作**策略**，用 RL **最大化奖励模型给的分数**——同时加一个**KL 惩罚**防止它偏离 SFT 模型太远(否则会"钻空子"刷高分却胡言乱语)。
   **RL optimization (PPO):** treat the LM as a **policy**, use RL to **maximize the reward model's score** — with a **KL penalty** to keep it close to the SFT model (else it "games" the reward with gibberish).

本课聚焦最核心、最可实现的一步：**奖励模型(从偏好学习)**。
We focus on the most core, most implementable step: the **reward model (learning from preferences)**.


<a id="2"></a>
## 2. 奖励模型：从人类偏好学习（从零）⭐ / Reward Model From Scratch

**关键洞察**：让人**直接给回答打分**(打 7.5 分还是 8 分?)很难、很主观；但让人**在两个回答里选哪个更好**容易得多、也更一致。所以 RLHF 用**成对偏好(pairwise preference)**：对每个 prompt，收集 (chosen 较好, rejected 较差) 对。
**Key insight:** asking humans to **score answers directly** (7.5 or 8?) is hard and subjective; asking which of **two answers is better** is far easier and more consistent. So RLHF uses **pairwise preferences**: per prompt, collect (chosen better, rejected worse) pairs.

奖励模型 $r_\theta$ 给每个回答输出一个标量分数，用 **Bradley-Terry 偏好损失**训练：让 chosen 的分数**高于** rejected。
The reward model $r_\theta$ outputs a scalar score per answer, trained with the **Bradley-Terry preference loss**: make chosen score **higher than** rejected.

$$\mathcal{L} = -\log \sigma\big(r_\theta(\text{chosen}) - r_\theta(\text{rejected})\big)$$

直觉：最小化它 ⇔ 拉大 (chosen − rejected) 的分差。下面用**合成偏好数据**从零训练一个奖励模型(把回答表示成特征向量, 隐含一个"真实质量")。
Intuition: minimizing it ⇔ widening the (chosen − rejected) score gap. Below we train a reward model from scratch on **synthetic preference data** (responses as feature vectors with a hidden "true quality").


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
sns.set_theme(style="whitegrid"); torch.manual_seed(0); np.random.seed(0)

# 合成: 每个"回答"是一个特征向量, 真实质量 = 一组隐藏权重·特征 (人类偏好它) / synthetic responses
D = 8
true_w = torch.randn(D)                                   # 隐藏的"真实质量"权重(人类心里的标准) / hidden quality weights
def quality(x): return x @ true_w                         # 真实质量分(我们/人类知道, 模型不知道) / true quality
def make_prefs(n):
    A = torch.randn(n, D); B = torch.randn(n, D)           # 每对两个回答 / two responses per pair
    qa, qb = quality(A), quality(B)
    chosen = torch.where((qa > qb)[:, None], A, B)         # 质量高的=chosen / higher quality = chosen
    rejected = torch.where((qa > qb)[:, None], B, A)
    return chosen, rejected
chosen, rejected = make_prefs(3000); c_te, r_te = make_prefs(600)

reward_model = nn.Sequential(nn.Linear(D, 16), nn.ReLU(), nn.Linear(16, 1))   # 奖励模型: 回答→分数 / response→score
opt = torch.optim.Adam(reward_model.parameters(), 1e-2)
losses = []
for epoch in range(150):
    rc = reward_model(chosen).squeeze(); rr = reward_model(rejected).squeeze()
    loss = -F.logsigmoid(rc - rr).mean()                  # Bradley-Terry 偏好损失 / preference loss
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
# 评估: 在测试对上, 奖励模型给 chosen 打分更高的比例 / accuracy: RM scores chosen higher
with torch.no_grad():
    acc = (reward_model(c_te).squeeze() > reward_model(r_te).squeeze()).float().mean().item()
print(f"奖励模型训练完成, 偏好损失 {losses[0]:.3f} → {losses[-1]:.3f}")
print(f"测试集上'给 chosen 打分更高'的比例 = {acc:.3f}  (学会了人类偏好!)")
# 学到的奖励 vs 真实质量是否相关 / does learned reward correlate with true quality?
with torch.no_grad():
    X = torch.randn(500, D); learned = reward_model(X).squeeze().numpy(); true_q = quality(X).numpy()
corr = np.corrcoef(learned, true_q)[0,1]
fig, axes = plt.subplots(1,2,figsize=(11,3.6))
axes[0].plot(losses); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("偏好损失"); axes[0].set_title("奖励模型: Bradley-Terry 损失下降")
axes[1].scatter(true_q, learned, s=8, alpha=0.5); axes[1].set_xlabel("真实质量(人类标准)"); axes[1].set_ylabel("奖励模型打分")
axes[1].set_title(f"学到的奖励 vs 真实质量 (相关系数={corr:.2f})")
plt.tight_layout(); plt.show()
print(f"奖励模型打分与'真实质量'高度相关({corr:.2f}) → 仅从'两两比较'就学会了给回答打分")


<a id="3"></a>
## 3. PPO：用奖励优化策略 ⭐ / PPO: Optimizing the Policy with Reward

有了奖励模型，第三步用**强化学习**优化语言模型(策略)：让它生成的回答**奖励分数尽可能高**。用的算法是 **PPO(近端策略优化)**。目标(直觉版)：
With a reward model, step 3 uses **RL** to optimize the LM (policy) so its generations get **high reward**. The algorithm is **PPO (Proximal Policy Optimization)**. Objective (intuition):

$$\max_{\text{policy}}\ \mathbb{E}[\,r_\theta(\text{回答})\,] - \beta\, \mathrm{KL}(\text{policy} \,\|\, \text{SFT模型})$$

- 第一项：让回答的**奖励分数高**(人类更喜欢)。
  First term: make responses **score high** (humans prefer them).
- 第二项：**KL 惩罚**——别离 SFT 模型太远。**为什么必须有它**(面试)：奖励模型不完美，纯追求高分会让模型**"钻空子"(reward hacking)** ——输出一些骗过奖励模型但其实是垃圾的文本。KL 惩罚像缰绳，把策略拴在合理的语言分布附近。
  Second term: **KL penalty** — don't drift far from the SFT model. **Why essential** (interview): the reward model is imperfect; chasing pure score leads to **reward hacking** — text that fools the RM but is garbage. The KL penalty is a leash keeping the policy near sensible language.

**整体直觉**：SFT 教模型"怎么回答"，奖励模型刻画"什么是好回答"，PPO 让模型**朝好回答的方向调整**(但别跑偏)。RLHF 训练复杂、不稳定、要同时跑 4 个模型(策略/参考/奖励/价值)——这也是 DPO 出现的动机。
**Overall:** SFT teaches "how to answer," the reward model captures "what's a good answer," PPO **nudges the model toward good answers** (without drifting). RLHF is complex, unstable, and runs 4 models (policy/reference/reward/value) — motivating DPO.


<a id="4"></a>
## 4. DPO：更简单的对齐 + 小结 ⭐ / DPO: Simpler Alignment

**DPO(直接偏好优化, Direct Preference Optimization, 2023)** 是近年的重要突破：**跳过"训练奖励模型 + 跑 PPO 强化学习"这两步**，直接用偏好对 (chosen, rejected) 优化语言模型本身。
**DPO (Direct Preference Optimization, 2023)** is a major recent breakthrough: **skip "train a reward model + run PPO RL"**, directly optimizing the LM on preference pairs (chosen, rejected).

DPO 的关键洞察：可以用数学推导证明，"最大化奖励 + KL 约束"的最优解，等价于一个**直接在偏好对上的简单分类损失**——本质是让模型**提高 chosen 的概率、降低 rejected 的概率**(相对一个冻结的参考模型)：
DPO's key insight: one can prove the optimal solution of "maximize reward + KL constraint" equals a **simple classification loss directly on preference pairs** — essentially **raise the probability of chosen, lower that of rejected** (relative to a frozen reference model):

$$\mathcal{L}_{DPO} = -\log\sigma\Big(\beta\big[\log\tfrac{\pi(\text{chosen})}{\pi_{ref}(\text{chosen})} - \log\tfrac{\pi(\text{rejected})}{\pi_{ref}(\text{rejected})}\big]\Big)$$

**DPO vs RLHF(面试高频)**：DPO **不需要单独的奖励模型、不需要 RL/PPO、更稳定、更省资源**，效果常与 RLHF 相当。所以现在很多开源对齐(如 Zephyr、很多 LLaMA 微调)都用 DPO。代价：它需要现成的偏好对数据，且灵活性不如完整 RLHF。
**DPO vs RLHF (high-frequency):** DPO needs **no separate reward model, no RL/PPO, is more stable and cheaper**, often matching RLHF. Hence many open alignments (Zephyr, many LLaMA fine-tunes) use DPO. Cost: it needs preference-pair data and is less flexible than full RLHF.

```
对齐问题: 基础模型只会续写, 不会"有用/诚实/无害"地回答 → 需用人类偏好对齐
RLHF三步: ①SFT(指令微调,学会回答格式) ②奖励模型(从两两偏好学打分) ③PPO(最大化奖励+KL惩罚)
奖励模型: 用成对偏好(chosen>rejected)+Bradley-Terry损失 -logσ(r_chosen-r_rejected); 比直接打分更可靠
PPO的KL惩罚: 防止reward hacking(钻奖励模型空子), 把策略拴在SFT附近
DPO: 跳过奖励模型+RL, 直接在偏好对上优化策略(等价最优解); 更简单稳定省资源, 现广泛使用
对齐 = 把基础模型变成ChatGPT式助手的关键; 还有 RLAIF(AI反馈)/Constitutional AI 等
```

### 💡 面试速查 / Interview cheat-sheet
1. **RLHF三步**: SFT → 奖励模型(偏好学习) → PPO(最大化奖励+KL)。
   RLHF: SFT → reward model (preference learning) → PPO (max reward + KL).
2. **奖励模型**: 成对偏好 + Bradley-Terry 损失; 两两比较比打分可靠。
   Reward model: pairwise preferences + Bradley-Terry loss; comparisons beat scoring.
3. **KL惩罚**: 防 reward hacking, 拴在 SFT 附近。
   KL penalty: prevents reward hacking, leashes to SFT.
4. **DPO**: 跳过奖励模型+RL, 直接优化偏好; 更稳更省, 常与RLHF相当。
   DPO: skip RM+RL, optimize preferences directly; stabler/cheaper, often matches RLHF.
5. **对齐目的**: helpful/honest/harmless; 把基础模型变成听话助手。
   Alignment goal: helpful/honest/harmless; turn a base model into an assistant.

### 下一节 / Next
**12.11 Prompt 工程**——不改模型权重, 仅靠**怎么写提示词**就能大幅改变 LLM 的表现。few-shot、思维链(CoT)、ReAct、自洽等技巧是用好 LLM 的"软技能", 也是最便宜的适配方式。
**12.11 Prompt Engineering** — without changing weights, **how you write the prompt** dramatically changes LLM behavior. Few-shot, chain-of-thought (CoT), ReAct, self-consistency are the cheapest way to adapt LLMs.
